# 02 — Preprocessing

This notebook demonstrates the audio preprocessing pipeline:

1. `load_audio` — load any audio file, resample to 22 050 Hz, mono.
2. `normalise_audio` — peak-normalise to [-1, 1].
3. `trim_silence` — strip leading/trailing silence.
4. Before / after spectrograms.
5. `segment_fixed` — slice into 3-second clips and display all of them.

> If real data is not available, a synthetic signal is used instead.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.audio_utils import (
    load_audio, normalise_audio, trim_silence,
    compute_log_mel_spectrogram,
)
from src.segmentation import segment_fixed

SR = 22050
MANIFEST_PATH = os.path.join(PROJECT_ROOT, "data", "manifest.csv")

print(f"Project root: {PROJECT_ROOT}")

## 1. Pick a sample file (or synthesise one)

In [ ]:
sample_path = None

try:
    df = pd.read_csv(MANIFEST_PATH)
    manifest_dir = os.path.dirname(MANIFEST_PATH)
    for _, row in df.iterrows():
        candidate = os.path.join(manifest_dir, row["clip_path"])
        if os.path.isfile(candidate):
            sample_path = candidate
            break
except FileNotFoundError:
    pass

if sample_path:
    print(f"Using real clip: {sample_path}")
    raw, sr = load_audio(sample_path, target_sr=SR)
else:
    print("⚠️  No data files — using a synthetic signal.")
    sr = SR
    t = np.linspace(0, 5.0, int(SR * 5.0), endpoint=False)
    # Create a chirp with silence padding
    chirp = 0.4 * np.sin(2 * np.pi * (200 + 200 * t) * t).astype(np.float32)
    raw = np.concatenate([np.zeros(SR, dtype=np.float32), chirp,
                          np.zeros(SR, dtype=np.float32)])

print(f"Raw signal: {len(raw)} samples, {len(raw)/sr:.2f} s")

## 2. Demonstrate `load_audio`, `normalise_audio`, `trim_silence`

In [ ]:
normed = normalise_audio(raw)
trimmed = trim_silence(normed, sr, top_db=30)

print(f"Raw      : {len(raw):>8d} samples  peak={np.max(np.abs(raw)):.4f}")
print(f"Normalised: {len(normed):>8d} samples  peak={np.max(np.abs(normed)):.4f}")
print(f"Trimmed  : {len(trimmed):>8d} samples  ({len(trimmed)/sr:.2f} s)")

## 3. Before / after spectrograms

In [ ]:
spec_raw = compute_log_mel_spectrogram(raw, sr)
spec_trimmed = compute_log_mel_spectrogram(trimmed, sr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

im1 = ax1.imshow(spec_raw, aspect="auto", origin="lower", cmap="magma")
ax1.set_title("Before (raw)")
ax1.set_xlabel("Frame")
ax1.set_ylabel("Mel bin")
fig.colorbar(im1, ax=ax1, format="%+2.0f dB")

im2 = ax2.imshow(spec_trimmed, aspect="auto", origin="lower", cmap="magma")
ax2.set_title("After (normalised + trimmed)")
ax2.set_xlabel("Frame")
ax2.set_ylabel("Mel bin")
fig.colorbar(im2, ax=ax2, format="%+2.0f dB")

fig.tight_layout()
plt.show()

## 4. Segment into 3-second clips and display

In [ ]:
clips = segment_fixed(trimmed, sr, clip_duration=3.0, overlap=0.5)
print(f"Segmentation produced {len(clips)} clips "
      f"({len(clips[0])/sr:.1f} s each)")

n_show = min(len(clips), 6)
fig, axes = plt.subplots(n_show, 2, figsize=(12, 2.5 * n_show))
if n_show == 1:
    axes = axes.reshape(1, -1)

for i in range(n_show):
    clip = clips[i]
    t = np.arange(len(clip)) / sr

    axes[i, 0].plot(t, clip, linewidth=0.4, color="steelblue")
    axes[i, 0].set_title(f"Clip {i} — waveform", fontsize=9)
    axes[i, 0].set_xlabel("Time (s)")
    axes[i, 0].set_ylabel("Amp")

    spec = compute_log_mel_spectrogram(clip, sr)
    axes[i, 1].imshow(spec, aspect="auto", origin="lower", cmap="magma")
    axes[i, 1].set_title(f"Clip {i} — spectrogram", fontsize=9)
    axes[i, 1].set_xlabel("Frame")
    axes[i, 1].set_ylabel("Mel bin")

fig.tight_layout()
plt.show()